In [1]:

from main import create_gui

gui = create_gui('my_model16082025.h5')

In [2]:
import re

raw = open("requirements_raw.txt", encoding="utf-8").read().splitlines()

out = []
for s in raw:
    s = s.strip()
    if not s or s.startswith("#"):
        continue
    # 云端不可复现的本地路径/URL/可编辑安装
    if " @ file:" in s or " @ http" in s or " @ https" in s or s.startswith("-e "):
        continue
    out.append(s)

# Windows-only 包，Cloud/Linux 会炸（可按需增减）
win_only = {"pywin32", "pywinpty", "win-inet-pton"}
out2 = []
for s in out:
    name = re.split(r"[=<>!~]", s, 1)[0].strip().lower()
    if name in win_only:
        continue
    out2.append(s)

open("requirements.txt", "w", encoding="utf-8").write("\n".join(out2) + "\n")
print("Wrote requirements.txt with", len(out2), "packages")


Wrote requirements.txt with 73 packages


In [1]:
import streamlit as st

st.set_page_config(page_title="Private Demo", page_icon="🔒", layout="centered")

st.title("🔒 Streamlit 私有仓库 Demo")
st.write("如果你能看到这页，说明部署成功。")

# 演示 secrets（可选：没有配 secrets 也能跑）
token = st.secrets.get("DEMO_TOKEN", "（未配置 secrets）")
st.info(f"DEMO_TOKEN: {token}")

name = st.text_input("输入你的名字：", placeholder="例如：Alice")
if st.button("打招呼"):
    st.success(f"你好，{name or '陌生人'}！")


2026-01-23 14:09:52.943 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-23 14:09:52.945 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-23 14:09:53.516 
  command:

    streamlit run C:\Users\xi_b\AppData\Local\miniforge3\envs\gems\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-01-23 14:09:53.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-23 14:09:53.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-23 14:09:53.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-23 14:09:53.520 Thread 'MainThread': missing ScriptRunContext! This warning 

StreamlitSecretNotFoundError: No secrets found. Valid paths for a secrets.toml file or secret directories are: C:\Users\xi_b\.streamlit\secrets.toml, C:\Users\xi_b\3D Objects\web development\web design\2201-2026\wed claude\.streamlit\secrets.toml

In [1]:
pip install fpdf2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import warnings
import sys
import time

# 设置页面配置 (必须是第一个 Streamlit 命令)
st.set_page_config(
    page_title="Concrete Mix AI Optimizer",
    page_icon="🏗️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# 抑制警告
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

# 引入你的自定义模块 (假设文件在同一目录下)
# 如果没有这些文件，请确保将原本的逻辑类粘贴进来或放在对应文件中
try:
    from model_wrapper import ModelWrapper
    from metrics import MetricsCalculator
    from nsga_problem import ConcreteMixProblem
except ImportError:
    st.error("⚠️ 缺少必要的模块 (model_wrapper, metrics, nsga_problem)。请确保这些文件在同一目录下。")
    st.stop()

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

# ==========================================
# 🎨 1. 高级 UI 设计系统 (CSS 注入)
# ==========================================
class DesignSystem:
    # 配色方案
    COLORS = {
        "primary": "#2563EB",       # Royal Blue
        "secondary": "#1E293B",     # Slate 800
        "accent": "#06B6D4",        # Cyan
        "background": "#F8FAFC",    # Slate 50
        "card_bg": "#FFFFFF",
        "text_dark": "#0F172A",
        "text_light": "#64748B",
        "success": "#10B981",
        "warning": "#F59E0B",
        "danger": "#EF4444",
    }

    @staticmethod
    def inject_css():
        st.markdown(f"""
        <style>
            /* 全局背景和字体 */
            .stApp {{
                background-color: {DesignSystem.COLORS['background']};
                font-family: 'Inter', sans-serif;
            }}
            
            /* 标题样式 */
            h1, h2, h3 {{
                color: {DesignSystem.COLORS['secondary']};
                font-weight: 700;
            }}
            
            /* 自定义 Metric 卡片 */
            div[data-testid="metric-container"] {{
                background-color: {DesignSystem.COLORS['card_bg']};
                border: 1px solid #E2E8F0;
                padding: 15px;
                border-radius: 10px;
                box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
                transition: all 0.2s;
            }}
            div[data-testid="metric-container"]:hover {{
                transform: translateY(-2px);
                box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1);
                border-color: {DesignSystem.COLORS['primary']};
            }}
            
            /* 侧边栏样式 */
            section[data-testid="stSidebar"] {{
                background-color: #FFFFFF;
                border-right: 1px solid #E2E8F0;
            }}
            
            /* 按钮美化 */
            div.stButton > button {{
                background: linear-gradient(135deg, {DesignSystem.COLORS['primary']} 0%, #1D4ED8 100%);
                color: white;
                border: none;
                padding: 0.6rem 1.2rem;
                border-radius: 8px;
                font-weight: 600;
                box-shadow: 0 4px 6px -1px rgba(37, 99, 235, 0.2);
                transition: all 0.2s;
            }}
            div.stButton > button:hover {{
                transform: translateY(-1px);
                box-shadow: 0 6px 8px -1px rgba(37, 99, 235, 0.3);
            }}
            div.stButton > button:active {{
                transform: translateY(0px);
            }}

            /* Tab 样式 */
            .stTabs [data-baseweb="tab-list"] {{
                gap: 10px;
            }}
            .stTabs [data-baseweb="tab"] {{
                height: 50px;
                white-space: pre-wrap;
                background-color: #FFFFFF;
                border-radius: 8px;
                color: {DesignSystem.COLORS['text_light']};
                font-weight: 600;
                border: 1px solid #E2E8F0;
            }}
            .stTabs [data-baseweb="tab"][aria-selected="true"] {{
                background-color: {DesignSystem.COLORS['primary']} !important;
                color: white !important;
                border: none;
            }}
            
            /* Expander 样式 */
            .streamlit-expanderHeader {{
                background-color: white;
                border-radius: 8px;
                border: 1px solid #E2E8F0;
            }}
        </style>
        """, unsafe_allow_html=True)

# ==========================================
# ⚙️ 2. 初始化与配置
# ==========================================

# 注入 CSS
DesignSystem.inject_css()

# 常量定义
FIXED_WC = 0.5
FIXED_GYPSUM = 4.0
FIXED_TEMP = 25.0
TOTAL_BINDER_TARGET = 100.0 - FIXED_GYPSUM
MATERIALS = [
    ("C3S", "C3S", 0.82, 0.141),
    ("C2S", "C2S", 0.69, 0.141),
    ("C3A", "C3A", 0.73, 0.141),
    ("C4AF", "C4AF", 0.55, 0.141),
    ("Silica fume", "silica_fume", 0.0035, 4.92),
    ("GGBFS", "GGBFS", 0.13, 0.056),
    ("Fly ash", "fly_ash", 0.004, 0.02),
    ("Calcined clay", "calcined_clay", 0.27, 0.11),
    ("Limestone", "limestone", 0.0023, 0.0227),
    ("Gypsum", "Gypsum", 0.0082, 0.051),
]

# 加载模型 (使用缓存避免重复加载)
@st.cache_resource
def load_resources(model_path):
    try:
        model = ModelWrapper(model_path)
        metrics = MetricsCalculator(FIXED_WC, FIXED_GYPSUM, FIXED_TEMP)
        return model, metrics
    except Exception as e:
        return None, None

# 假设模型路径
MODEL_PATH = "my_model16082025.h5" # 请修改为你的实际路径
if not os.path.exists(MODEL_PATH):
    # 这里只是为了防止报错，如果文件不存在，用户需要自己指定
    # 在实际使用中，可以添加文件上传功能
    pass 

model, metrics = load_resources(MODEL_PATH)

if model is None:
    st.warning(f"⚠️ 模型未找到: `{MODEL_PATH}`。请确保模型文件存在，或在代码中修改路径。")
    # 为了演示 UI，即使没有模型也继续渲染，但在点击运行时会报错
    
# 初始化 Session State
if "df_pareto" not in st.session_state:
    st.session_state.df_pareto = None
if "run_status" not in st.session_state:
    st.session_state.run_status = "idle"

# ==========================================
# 🖥️ 3. 侧边栏：材料定义
# ==========================================
with st.sidebar:
    st.markdown("### 🏗️ Material Inputs")
    st.info(f"Fixed Params: w/c={FIXED_WC}, T={FIXED_TEMP}°C")
    
    with st.expander("🔶 Clinker Phases", expanded=True):
        c3s_rng = st.slider("C3S %", 0.0, 100.0, (45.0, 80.0), step=0.5)
        c2s_rng = st.slider("C2S %", 0.0, 100.0, (10.0, 32.0), step=0.5)
        c3a_rng = st.slider("C3A %", 0.0, 100.0, (0.0, 14.0), step=0.5)
        c4af_rng = st.slider("C4AF %", 0.0, 100.0, (0.0, 15.0), step=0.5)
        cl_sum_rng = st.slider("Clinker Total %", 0.0, 100.0, (20.0, 96.0), step=0.5)

    with st.expander("🟢 SCM Ranges", expanded=False):
        sf_rng = st.slider("Silica Fume %", 0.0, 50.0, (0.0, 10.0), step=0.5)
        gg_rng = st.slider("GGBFS %", 0.0, 100.0, (0.0, 80.0), step=0.5)
        fa_rng = st.slider("Fly Ash %", 0.0, 100.0, (0.0, 35.0), step=0.5)
        cc_rng = st.slider("Calcined Clay %", 0.0, 100.0, (0.0, 35.0), step=0.5)
        ls_rng = st.slider("Limestone %", 0.0, 100.0, (0.0, 35.0), step=0.5)

    st.markdown("### ⏱️ Curing")
    time_val = st.number_input("Time (days)", min_value=1, max_value=36500, value=28, step=1)
    
    # 快捷时间按钮
    cols = st.columns(4)
    if cols[0].button("7d"): time_val = 7
    if cols[1].button("28d"): time_val = 28
    if cols[2].button("90d"): time_val = 90
    if cols[3].button("1y"): time_val = 365

# ==========================================
# 🚀 4. 主界面
# ==========================================

# 顶部 Header
st.markdown(f"""
    <div style="padding-bottom: 20px;">
        <h1 style="margin-bottom:0px;">Concrete Mix AI Optimizer</h1>
        <p style="color:{DesignSystem.COLORS['text_light']}; font-size:16px;">
            NSGA-II Multi-Objective Optimization for Sustainable Construction
        </p>
    </div>
""", unsafe_allow_html=True)

# Tabs 分区
tab_config, tab_run, tab_compare = st.tabs(["⚙️ Settings & Factors", "▶️ Run Optimization", "⚖️ Analysis & Comparison"])

# --- TAB 1: 配置 ---
with tab_config:
    col1, col2 = st.columns([1, 1], gap="large")
    
    with col1:
        st.markdown("#### 🎯 Optimization Objectives")
        st.caption("Select at least one objective to optimize.")
        
        # 使用卡片包裹 Checkbox
        with st.container():
            st.markdown(f"""<div style="background:white; padding:15px; border-radius:10px; border:1px solid #E2E8F0;">""", unsafe_allow_html=True)
            obj_e_max = st.checkbox("📈 Maximize Elastic Modulus (E)", value=True)
            obj_co2_min = st.checkbox("♻️ Minimize CO₂ Emission", value=False)
            obj_cost_min = st.checkbox("💰 Minimize Cost", value=False)
            obj_net_min = st.checkbox("🌍 Minimize Net Emission (CO₂ - Uptake)", value=True)
            obj_co2abs_max = st.checkbox("🌱 Maximize CO₂ Uptake", value=False)
            st.markdown("</div>", unsafe_allow_html=True)

        st.markdown("#### 🧬 NSGA-II Parameters")
        use_custom_nsga = st.toggle("Use Custom Parameters", value=False)
        
        if use_custom_nsga:
            ga_pop = st.number_input("Population Size", 40, 500, 100, step=10)
            ga_gen = st.number_input("Generations", 20, 500, 50, step=10)
            ga_seed = st.number_input("Random Seed", 0, 999999, 42)
        else:
            st.info("Using defaults: Pop=100, Gen=50, Seed=42")
            ga_pop, ga_gen, ga_seed = 100, 50, 42

    with col2:
        st.markdown("#### 🏭 Material Factors (CO₂ & Cost)")
        st.caption("Adjust the environmental impact and cost per kg for each material.")
        
        # 使用 Tabs 在 Factors 内部切换，节省空间
        fac_tab1, fac_tab2 = st.tabs(["CO₂ Emission (kg/kg)", "Cost (€/kg)"])
        
        factors_co2 = {}
        factors_cost = {}
        
        with fac_tab1:
            for label, key, emis, cost in MATERIALS:
                factors_co2[key] = st.slider(f"{label}", 0.0, 2.0, float(emis), 0.001, key=f"co2_{key}")
        
        with fac_tab2:
            for label, key, emis, cost in MATERIALS:
                factors_cost[key] = st.slider(f"{label}", 0.0, 10.0, float(cost), 0.01, key=f"cost_{key}")

# --- TAB 2: 运行优化 ---
with tab_run:
    st.markdown("### 🚀 Execute Optimization")
    
    # 构造运行按钮区域
    col_btn, col_status = st.columns([1, 4])
    with col_btn:
        run_btn = st.button("Start Simulation", type="primary", use_container_width=True)
    
    # 状态显示
    status_placeholder = st.empty()
    
    if run_btn:
        if model is None:
            st.error("Model not loaded. Cannot run optimization.")
        else:
            # 1. 收集目标
            objectives = []
            if obj_e_max: objectives.append("E_max")
            if obj_co2_min: objectives.append("CO2_min")
            if obj_cost_min: objectives.append("Cost_min")
            if obj_net_min: objectives.append("Net_min")
            if obj_co2abs_max: objectives.append("CO2abs_max")
            
            if not objectives:
                st.error("Please select at least one objective in Settings.")
            else:
                # 2. 准备问题
                status_placeholder.info("⏳ Initializing NSGA-II Algorithm...")
                progress_bar = st.progress(0)
                
                clinker_bounds = {
                    "C3S": c3s_rng, "C2S": c2s_rng, "C3A": c3a_rng, "C4AF": c4af_rng
                }
                scms_bounds = {
                    "silica_fume": sf_rng, "GGBFS": gg_rng, "fly_ash": fa_rng, 
                    "calcined_clay": cc_rng, "limestone": ls_rng
                }
                
                problem = ConcreteMixProblem(
                    model=model,
                    metrics_calc=metrics,
                    clinker_bounds=clinker_bounds,
                    scms_bounds=scms_bounds,
                    clinker_sum_rng=cl_sum_rng,
                    total_binder_target=TOTAL_BINDER_TARGET,
                    time_val=int(time_val),
                    emission_factors=factors_co2,
                    cost_factors=factors_cost,
                    objectives=objectives
                )
                
                # 3. 运行算法
                algorithm = NSGA2(
                    pop_size=ga_pop,
                    sampling=FloatRandomSampling(),
                    crossover=SBX(prob=0.9, eta=15.0),
                    mutation=PM(prob=0.1, eta=20.0),
                    eliminate_duplicates=True
                )
                
                # 模拟进度条更新 (pymoo 没有原生回调更新进度条，这里只能是个视觉反馈)
                for i in range(1, 101, 20):
                    time.sleep(0.05)
                    progress_bar.progress(i)
                    status_placeholder.markdown(f"🧬 Evolving Generation... ({i}%)")

                res = minimize(
                    problem, algorithm, 
                    termination=get_termination("n_gen", ga_gen), 
                    seed=ga_seed, 
                    verbose=False
                )
                
                progress_bar.progress(100)
                
                # 4. 处理结果
                X = res.X
                mixes = problem.decode(X)
                preds = model.predict(mixes)
                df_result = metrics.add_metrics(mixes, preds[:, 0], preds[:, 1], factors_co2, factors_cost)
                
                # 保存到 Session State
                st.session_state.df_pareto = df_result
                st.session_state.run_status = "success"
                status_placeholder.success(f"✅ Optimization Completed! Found {len(df_result)} Pareto solutions.")

    # 结果展示区域 (仅当有结果时显示)
    if st.session_state.df_pareto is not None:
        df = st.session_state.df_pareto
        st.markdown("---")
        
        # 结果分栏：图表 + 表格
        res_col1, res_col2 = st.columns([3, 2])
        
        with res_col1:
            st.markdown("#### 📊 Pareto Front Visualization")
            
            # 智能选择坐标轴
            available_cols = df.columns
            y_axis = "E" if "E" in available_cols else available_cols[-1]
            x_axis = "Net_emission" if "Net_emission" in available_cols else (
                "CO2_emission" if "CO2_emission" in available_cols else available_cols[-2]
            )
            c_axis = "Cost" if "Cost" in available_cols else None
            
            # 允许用户覆盖
            c1, c2, c3 = st.columns(3)
            x_sel = c1.selectbox("X Axis", df.columns, index=list(df.columns).index(x_axis))
            y_sel = c2.selectbox("Y Axis", df.columns, index=list(df.columns).index(y_axis))
            c_sel = c3.selectbox("Color", [None] + list(df.columns), index=list(df.columns).index(c_axis) if c_axis else 0)
            
            # Plotly 图表
            fig = px.scatter(
                df, x=x_sel, y=y_sel, color=c_sel,
                size="C3S", hover_data=["silica_fume", "GGBFS", "fly_ash"],
                title=f"{y_sel} vs {x_sel}",
                template="plotly_white",
                color_continuous_scale="Viridis"
            )
            fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)",
                paper_bgcolor="rgba(0,0,0,0)",
                font=dict(family="Inter", size=12, color=DesignSystem.COLORS['text_dark']),
                height=450
            )
            st.plotly_chart(fig, use_container_width=True)

        with res_col2:
            st.markdown("#### 📋 Solution Data")
            st.dataframe(
                df.style.format("{:.2f}").background_gradient(cmap="Blues", subset=["E"]), 
                height=400,
                use_container_width=True
            )
            
            # 下载按钮
            csv = df.to_csv(index=False).encode('utf-8')
            st.download_button(
                label="📥 Download Results (CSV)",
                data=csv,
                file_name="pareto_optimization_results.csv",
                mime="text/csv",
                use_container_width=True
            )

# --- TAB 3: 对比分析 ---
with tab_compare:
    col_setup, col_viz = st.columns([1, 2], gap="large")
    
    with col_setup:
        st.markdown("#### ⚖️ Reference Material (OPC)")
        with st.container():
            st.markdown(f"""<div style="background:white; padding:15px; border-radius:10px; border:1px solid #E2E8F0; margin-bottom:15px;">""", unsafe_allow_html=True)
            ref_name = st.text_input("Reference Name", "Standard OPC")
            ref_E = st.number_input("Ref E (GPa)", value=30.0, step=0.5)
            ref_CO2 = st.number_input("Ref CO₂ (kg/kg)", value=0.85, step=0.01)
            ref_Cost = st.number_input("Ref Cost (€/kg)", value=0.10, step=0.01)
            ref_Net = st.number_input("Ref Net Emission (kg/kg)", value=0.85, step=0.01)
            st.markdown("</div>", unsafe_allow_html=True)

        st.markdown("#### Select Pareto Solution")
        if st.session_state.df_pareto is not None:
            df = st.session_state.df_pareto
            # 创建一个可读的标签列表
            mix_options = [f"Mix #{i}: E={row['E']:.1f}, Net={row['Net_emission']:.2f}" for i, row in df.iterrows()]
            selected_mix_str = st.selectbox("Choose a mix to compare:", mix_options)
            selected_idx = mix_options.index(selected_mix_str)
        else:
            st.warning("Please run optimization first.")
            selected_idx = None

    with col_viz:
        if selected_idx is not None and st.session_state.df_pareto is not None:
            row = st.session_state.df_pareto.iloc[selected_idx]
            
            st.markdown(f"### 🆚 Comparison: {ref_name} vs Pareto Mix #{selected_idx}")
            
            # 1. Delta Metrics (Kpi Cards)
            m1, m2, m3, m4 = st.columns(4)
            
            def get_delta(new, old, inverse=False):
                val = new - old
                pct = (val / old * 100) if old != 0 else 0
                return val, pct

            # E (越高越好)
            d_e, p_e = get_delta(row.get("E", 0), ref_E)
            m1.metric("Elastic Modulus", f"{row.get('E', 0):.2f} GPa", f"{d_e:+.2f} ({p_e:+.1f}%)")
            
            # Net Emission (越低越好 -> inverse=True for color logic, but Streamlit metric handles delta color automatically based on sign. 
            # We usually want Green for negative delta in CO2. Streamlit default: Green = Positive.
            # So we use delta_color="inverse" for bad-is-higher metrics)
            d_net, p_net = get_delta(row.get("Net_emission", 0), ref_Net)
            m2.metric("Net Emission", f"{row.get('Net_emission', 0):.2f}", f"{d_net:+.2f} ({p_net:+.1f}%)", delta_color="inverse")
            
            d_cost, p_cost = get_delta(row.get("Cost", 0), ref_Cost)
            m3.metric("Cost", f"€{row.get('Cost', 0):.2f}", f"{d_cost:+.2f} ({p_cost:+.1f}%)", delta_color="inverse")
            
            d_co2, p_co2 = get_delta(row.get("CO2_emission", 0), ref_CO2)
            m4.metric("Gross CO₂", f"{row.get('CO2_emission', 0):.2f}", f"{d_co2:+.2f} ({p_co2:+.1f}%)", delta_color="inverse")
            
            st.markdown("---")
            
            # 2. Bar Chart Comparison
            metrics_data = {
                "Metric": ["E (GPa)", "E (GPa)", "Net CO₂", "Net CO₂", "Cost", "Cost"],
                "Type": [ref_name, "Pareto Mix", ref_name, "Pareto Mix", ref_name, "Pareto Mix"],
                "Value": [ref_E, row.get("E", 0), ref_Net, row.get("Net_emission", 0), ref_Cost, row.get("Cost", 0)]
            }
            df_comp = pd.DataFrame(metrics_data)
            
            bar_fig = px.bar(
                df_comp, x="Metric", y="Value", color="Type", barmode="group",
                color_discrete_map={ref_name: "#94A3B8", "Pareto Mix": DesignSystem.COLORS['primary']},
                template="plotly_white"
            )
            bar_fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)",
                font=dict(family="Inter"),
                height=350,
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
            )
            st.plotly_chart(bar_fig, use_container_width=True)
            
        else:
            st.info("👈 Run optimization and select a mix to see comparison.")

HTML(value="<h1 style='color:#2c3e50; text-align:center;'>🏗️ Intelligent Concrete Mix Designer</h1>")